# Sample a small problem with the OpenJij Adapter

Collect candidates with OpenJij's simulated annealing and read their OMMX objective values and feasibility. **To choose a constraint penalty weight yourself, put it in a Policy, prepare the Instance, and call `sample_without_preparation()`.**

This page is self-contained if you know Python and basic variables, objectives, and constraints. Install SDK `3.0.0b7` using steps 1–2 of the [installation guide](../getting_started/install.md), then {ref}`add the OpenJij Adapter <install-openjij-adapter>`. Select that environment as your notebook kernel. No other Adapter or external data is needed.

## 1. Model an exactly-one choice

Choose exactly one of two options and maximize its score. An `Instance` collects variables, an objective, and constraints. `new_binary()` creates a variable taking 0 or 1, and `OneHotConstraint` requires exactly one variable to equal 1.


In [1]:
from ommx import FixedPenaltyPreparation, Instance, OneHotConstraint
from ommx_openjij_adapter import OMMXOpenJijSAAdapter

instance = Instance.maximize()
x = [instance.new_binary("x", subscripts=[i]) for i in range(2)]
instance.objective = 3 * x[0] + 2 * x[1]
instance.add_one_hot_constraint(OneHotConstraint(variables=x))

target = OMMXOpenJijSAAdapter.INPUT_CLASS
target.contains(instance)

False

`INPUT_CLASS` describes the problems the Adapter accepts directly. The OpenJij SA Adapter requires used variables to be Binary, a polynomial minimization objective, and no active constraints. Polynomial degree is not limited to two.

This Instance maximizes and has a OneHot constraint, so `contains()` is `False`. Prepare a representation OpenJij can search.

## 2. Choose a weight and prepare the Instance

Preparation transforms an Instance toward a target class using a Policy's allowed operations. Start with the Adapter's recommendation and add a **penalty weight** that makes constraint violations less attractive in the search objective. The OpenJij recommendation leaves this choice to the caller.


In [2]:
policy = OMMXOpenJijSAAdapter.recommended_preparation_policy()
policy.fixed_penalty = (
    FixedPenaltyPreparation.uniform_penalty_method_with_fixed_weight(weight=4.0)
)
instance.prepare(target, policy)

target.contains(instance)

True

Here Preparation lowers OneHot to an equality, converts maximization to minimization, and adds an equality penalty with weight 4. Selecting both options scores 5 but incurs a penalty of 4, making one option scoring 3 preferable. This weight is suitable for this small example, not necessarily for another problem.

`prepare()` changes the same Instance in place. `True` establishes its representation meets the input conditions; it does not guarantee candidates will satisfy the original constraints or be globally optimal.

## 3. Collect candidates from the prepared Instance

`sample_without_preparation()` uses the prepared Instance without additional Preparation. `num_reads` controls the number of trials and `seed` configures randomness.


In [3]:
sample_set = OMMXOpenJijSAAdapter.sample_without_preparation(
    instance, num_reads=10, seed=42
)
sample_set.summary

,objective,feasible
sample_id,,
5,3.0,True
4,3.0,True
7,3.0,True
6,3.0,True
1,3.0,True
0,3.0,True
3,3.0,True
2,3.0,True
9,3.0,True


The returned `SampleSet` evaluates the collected candidates against the OMMX problem. Each row identifies a candidate by `sample_id`; several rows can have identical variable values. Feasible candidates appear first, ordered by their reported objective. The sample composition may vary by environment.

- `objective`: here the score `3*x[0] + 2*x[1]` before adding the penalty. Compare it in the original maximization sense.
- `feasible`: here the check includes the retained OneHot constraint. Passing an unconstrained model to OpenJij does not establish original feasibility.

OpenJij's search energy differs from this reported objective. Objective-sense conversion, penalties, and omitted constants can make their values differ. This `summary` is not a raw-energy table.

## 4. Select among the returned candidates

`best_feasible` returns the candidate with the best reported objective **among the feasible candidates obtained**, as a `Solution`. `name` and `subscripts` are the model's variable labels; `value` is the candidate's assignment.


In [4]:
solution = sample_set.best_feasible
print(f"{solution.objective=}, {solution.feasible=}")
solution.decision_variables_df()[["name", "subscripts", "value"]]

solution.objective=3.0, solution.feasible=True


,name,subscripts,value
id,,,
0,x,[0],1.0
1,x,[1],0.0


If there is no feasible candidate, `best_feasible` raises an error. Inspect `feasible` and reconsider settings such as the weight and number of trials. **To change the weight, rerun from the model-building cell.** An already-prepared Instance belongs to the input class, so editing a Policy and calling `prepare()` does not replace the applied weight. Selecting a candidate obtained by annealing does not prove global optimality.

## Related explanations and an application

The usual `sample()` gets a new recommended Policy for that call. It does not accept the edited `policy` above, so pair `prepare()` with `sample_without_preparation()` when choosing your own weight.

- Input conditions and transformation choices: [InstanceClass](../user_guide/instance_class.md), [PreparationPolicy](../user_guide/preparation_policy.md).
- Result interpretation: [SampleSet](../user_guide/sample_set.md), [Transformed problems and removed constraints](../user_guide/removed_constraints.md).
- Exact penalty formula and conditions: {meth}`~ommx.FixedPenaltyPreparation.uniform_penalty_method_with_fixed_weight`. Do not confuse this equality penalty with the treatment of general inequalities.
- Optional application: [Sample TSP routes with OpenJij](../tutorial/tsp_sampling_with_openjij_adapter). This is not required for the small example.
